# 02 · Análise exploratória (EDA)

PIB dos municípios brasileiros (2021): distribuição, composição setorial, especialização, perfis e quadrantes.

## 0. Preparação

In [ ]:
# Habilitar recarga automática dos módulos
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")  # Apontar para a raiz do projeto

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.base import GRUPOS, carregar_municipios
from src.config import K_PERFIS, SETORES
from src.indicadores import (
    composicao_nacional,
    gini,
    hhi,
    participacoes,
    quociente_locacional,
)
from src.visual import CORES_QUADRANTES, CORES_SETORES, NOMES_SETORES, ORDEM_REGIOES

# Ajustar a exibição das tabelas
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.1f}".format)

# Carregar base municipal
mun = carregar_municipios()
print(mun.shape)

# Criar atalhos de participação e validade
PARTS = [f"part_{s}" for s in SETORES]
VALIDO = mun[PARTS].notna().all(axis=1)
print("municípios com participações válidas:", VALIDO.sum())

## 1. Panorama das variáveis

**Pergunta:** o município "típico" se parece com a média?

In [ ]:
# Resumir variáveis principais
cols = ["pib", "populacao", "pib_per_capita"]
print(mun[cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

# Comparar média e mediana
print(mun[cols].agg(["mean", "median"]))

# Medir assimetria
print(mun[cols].skew())

## 2. Concentração do PIB

**Pergunta:** quantos municípios concentram o PIB?

### 2.1 Participação acumulada

In [ ]:
# Ordenar municípios do maior para o menor PIB
ordenado = mun.sort_values("pib", ascending=False).reset_index(drop=True)

# Calcular participação acumulada no PIB total (%)
ordenado["pib_acum"] = ordenado["pib"].cumsum() / ordenado["pib"].sum() * 100

# Contar municípios necessários para atingir cada meta
for alvo in [25, 50, 80]:
    n = (ordenado["pib_acum"] < alvo).sum() + 1
    print(alvo, n, round(n / len(ordenado) * 100, 1))

# Mostrar os 10 maiores
print(ordenado[["nome", "pib", "pib_acum"]].head(10))

### 2.2 Índice de Gini

In [ ]:
# Comparar Gini das três variáveis
for col in ["pib", "populacao", "pib_per_capita"]:
    print(col, round(gini(mun[col]), 3))

### 2.3 Curva de Lorenz

In [ ]:
def curva_lorenz(x):
    """Calcular os pontos da curva de Lorenz."""
    # Ordenar valores em ordem crescente
    x = np.sort(np.asarray(x, dtype=float))

    # Calcular participação acumulada da variável (%)
    y = np.cumsum(x) / x.sum() * 100

    # Calcular fração acumulada de municípios (%)
    n = np.arange(1, len(x) + 1) / len(x) * 100

    # Incluir a origem (0, 0)
    return np.insert(n, 0, 0), np.insert(y, 0, 0)


# Criar figura com a linha de igualdade perfeita
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 100], y=[0, 100], name="Igualdade", line=dict(dash="dash", color="gray")
    )
)

# Adicionar curvas de Lorenz e imprimir a fatia dos 50% menores
for col, nome in [("pib", "PIB"), ("populacao", "População")]:
    x, y = curva_lorenz(mun[col])
    fig.add_trace(go.Scatter(x=x, y=y, name=nome, mode="lines"))
    print(nome, "- 50% menores municípios:", round(np.interp(50, x, y), 1), "%")

# Ajustar títulos e estilo
fig.update_layout(
    title="Curva de Lorenz: PIB e população dos municípios (2021)",
    xaxis_title="% acumulado de municípios (do menor ao maior)",
    yaxis_title="% acumulado da variável",
    template="plotly_white",
)
fig.show()

## 3. Forma das distribuições

**Pergunta:** como PIB, população e PIB per capita se distribuem?

In [ ]:
def hist_log(col, titulo, expoentes, rotulos, eixo):
    """Criar histograma da variável em escala logarítmica."""
    # Calcular logaritmo (base 10)
    df = pd.DataFrame({"log": np.log10(mun[col])})

    # Criar histograma
    fig = px.histogram(df, x="log", nbins=40, title=titulo, template="plotly_white")

    # Marcar a mediana
    fig.add_vline(
        x=np.log10(mun[col].median()), line_dash="dash", annotation_text="mediana"
    )

    # Rotular eixos com valores reais
    fig.update_xaxes(tickvals=expoentes, ticktext=rotulos, title=eixo)
    fig.update_yaxes(title="Número de municípios")
    return fig


# Exibir histograma do PIB (mil R$)
hist_log(
    "pib",
    "Distribuição do PIB municipal (2021)",
    [4, 5, 6, 7, 8, 9],
    ["R$ 10 mi", "R$ 100 mi", "R$ 1 bi", "R$ 10 bi", "R$ 100 bi", "R$ 1 tri"],
    "PIB (escala log)",
).show()

# Exibir histograma da população
hist_log(
    "populacao",
    "Distribuição da população municipal (2021)",
    [3, 4, 5, 6, 7],
    ["1 mil", "10 mil", "100 mil", "1 mi", "10 mi"],
    "População (escala log)",
).show()

# Exibir histograma do PIB per capita (R$)
hist_log(
    "pib_per_capita",
    "Distribuição do PIB per capita municipal (2021)",
    [4, 5, 6],
    ["R$ 10 mil", "R$ 100 mil", "R$ 1 mi"],
    "PIB per capita (escala log)",
).show()

## 4. PIB per capita por região

**Pergunta:** a renda municipal muda de região para região?

### 4.1 Boxplot

In [ ]:
# Resumir PIB per capita por região
resumo = mun.groupby("regiao")["pib_per_capita"].describe(percentiles=[0.25, 0.5, 0.75])
print(resumo.loc[ORDEM_REGIOES, ["count", "25%", "50%", "75%"]])

# Criar boxplot em escala logarítmica
fig = px.box(
    mun,
    x="regiao",
    y="pib_per_capita",
    log_y=True,
    category_orders={"regiao": ORDEM_REGIOES},
    points="outliers",
    title="PIB per capita municipal por região (2021)",
    template="plotly_white",
)
fig.update_xaxes(title="Região")
fig.update_yaxes(title="PIB per capita (R$, escala log)")
fig.show()

### 4.2 Histograma por região

In [ ]:
# Calcular logaritmo do PIB per capita
df = mun.assign(log_pc=np.log10(mun["pib_per_capita"]))

# Criar um histograma por região
fig = px.histogram(
    df,
    x="log_pc",
    facet_row="regiao",
    category_orders={"regiao": ORDEM_REGIOES},
    nbins=40,
    height=800,
    title="PIB per capita municipal por região (2021)",
    template="plotly_white",
)

# Rotular eixo com valores reais
fig.update_xaxes(
    tickvals=[4, 4.5, 5, 5.5, 6],
    ticktext=["R$ 10 mil", "R$ 32 mil", "R$ 100 mil", "R$ 316 mil", "R$ 1 mi"],
    title="PIB per capita (escala log)",
)
fig.update_yaxes(title="")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 5. Composição setorial

**Pergunta:** do que vive cada território?

### 5.1 Brasil, município típico e setor dominante

In [ ]:
# Calcular composição nacional (ponderada) e do município típico (mediana)
nacional = mun[SETORES].sum() / mun["vab_total"].sum() * 100
tipico = mun[PARTS].median()
comp = pd.DataFrame(
    {"nacional": nacional.values, "município típico": tipico.values},
    index=NOMES_SETORES,
)
print(comp.round(1))

# Identificar o setor dominante (só onde as participações são válidas)
dominante = mun.loc[VALIDO, PARTS].idxmax(axis=1).str.replace("part_vab_", "", regex=False)
print((dominante.value_counts(normalize=True) * 100).round(1))

### 5.2 Por região e por estado

In [ ]:
def composicao(df, por):
    """Calcular composição setorial (%) por grupo."""
    # Somar setores e VAB por grupo
    g = df.groupby(por)[SETORES + ["vab_total"]].sum()

    # Converter em % do VAB do grupo
    pct = g[SETORES].div(g["vab_total"], axis=0) * 100
    pct.columns = NOMES_SETORES
    return pct


def barras(pct, titulo, eixo):
    """Criar barras empilhadas da composição setorial."""
    fig = px.bar(
        pct,
        x=pct.index,
        y=NOMES_SETORES,
        color_discrete_map=CORES_SETORES,
        title=titulo,
        template="plotly_white",
        labels={"value": "% do VAB", "variable": "Setor"},
    )
    fig.update_xaxes(title=eixo)
    return fig


# Testar o efeito do DF no Centro-Oeste
co = mun[(mun["regiao"] == "Centro-Oeste") & (mun["uf_sigla"] != "DF")]
co_pct = co[SETORES].sum() / co["vab_total"].sum() * 100
co_pct.index = NOMES_SETORES
print(co_pct.round(1))

# Exibir composição por região
barras(
    composicao(mun, "regiao").loc[ORDEM_REGIOES],
    "Composição do VAB por região (2021)",
    "Região",
).show()

# Exibir composição por estado, ordenada pelo peso da adm. pública
uf_pct = composicao(mun, "uf_sigla").sort_values("Adm. pública")
barras(uf_pct, "Composição do VAB por estado (2021)", "Estado").show()

## 6. Administração pública, porte e região

**Pergunta:** a adm. pública pesa mais nos municípios pequenos?

### 6.1 Por faixa de população e por região

In [ ]:
# Definir faixas de população
limites = [0, 5_000, 10_000, 20_000, 50_000, 100_000, 500_000, np.inf]
rotulos = [
    "até 5 mil", "5 a 10 mil", "10 a 20 mil", "20 a 50 mil",
    "50 a 100 mil", "100 a 500 mil", "mais de 500 mil",
]
mun["faixa_pop"] = pd.cut(mun["populacao"], bins=limites, labels=rotulos)

# Marcar municípios em que a adm. pública é o maior setor (só onde válido)
mun["adm_dominante"] = False
dom = mun.loc[VALIDO, PARTS].idxmax(axis=1)
mun.loc[VALIDO, "adm_dominante"] = dom == "part_vab_adm_publica"

# Resumir por faixa de população
por_faixa = mun.groupby("faixa_pop", observed=True).agg(
    municipios=("cod", "count"),
    mediana_adm=("part_vab_adm_publica", "median"),
    pct_dominante=("adm_dominante", "mean"),
)
por_faixa["pct_dominante"] *= 100
print(por_faixa.round(1))

# Resumir por região
por_regiao = mun.groupby("regiao").agg(
    mediana_adm=("part_vab_adm_publica", "median"),
    pct_dominante=("adm_dominante", "mean"),
)
por_regiao["pct_dominante"] *= 100
print(por_regiao.loc[ORDEM_REGIOES].round(1))

### 6.2 Porte dentro de cada região

In [ ]:
# Agrupar a população em 3 portes
mun["porte"] = pd.cut(
    mun["populacao"],
    bins=[0, 20_000, 100_000, np.inf],
    labels=["até 20 mil", "20 a 100 mil", "mais de 100 mil"],
)

# Calcular a mediana da adm. pública por região e porte
tab = mun.pivot_table(
    index="regiao",
    columns="porte",
    values="part_vab_adm_publica",
    aggfunc="median",
    observed=True,
)
print(tab.loc[ORDEM_REGIOES].round(1))

# Contar municípios por região e porte
print(pd.crosstab(mun["regiao"], mun["porte"]).loc[ORDEM_REGIOES])

## 7. Especialização: HHI e quociente locacional

**Pergunta:** quem depende de um só setor e em que setor cada território é especializado?

### 7.1 HHI

In [ ]:
# Descrever o HHI (coluna já calculada na base)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(mun["hhi"].describe())
    print(mun.groupby("regiao")["hhi"].median().loc[ORDEM_REGIOES])

# Listar municípios de PIB acima de R$ 1 bilhão com maior concentração
cols = ["nome", "pib", "hhi"] + PARTS
print(mun[mun["pib"] > 1_000_000].nlargest(10, "hhi")[cols].round(2))

### 7.2 Checar a faixa teórica do HHI

In [ ]:
# Conferir faixa do HHI (esperado: entre 0 e 1) e quantidade de vazios
print(round(mun["hhi"].min(), 3), round(mun["hhi"].max(), 3), mun["hhi"].isna().sum())

# Listar municípios sem HHI e seus setores
print(mun.loc[mun["hhi"].isna(), ["nome", "vab_total"] + SETORES])

# Contar valores negativos por setor
print((mun[SETORES] < 0).sum())

### 7.3 Quociente locacional (QL)

In [ ]:
# Calcular a referência nacional e o QL dos municípios
ref = composicao_nacional(mun)
ql = quociente_locacional(mun, ref)

# Contar municípios especializados (QL > 1) por setor
print((ql > 1).sum())

# Calcular QL por região
reg = mun.groupby("regiao")[SETORES].sum()
print(quociente_locacional(reg, ref).loc[ORDEM_REGIOES].round(2))

### 7.4 Mapa de calor do QL por estado

In [ ]:
# Calcular o QL por estado
uf_soma = mun.groupby("uf_sigla")[SETORES].sum()
ql_uf = quociente_locacional(uf_soma, ref)
ql_uf.columns = NOMES_SETORES

# Exibir mapa de calor (centro da escala em QL = 1)
fig = px.imshow(
    ql_uf,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=1,
    aspect="auto",
    height=800,
    labels={"color": "QL"},
    title="Quociente locacional por estado (referência: Brasil, 2021)",
)
fig.update_xaxes(side="top")
fig.show()

# Listar o setor de maior especialização de cada estado
topo = pd.DataFrame({"setor": ql_uf.idxmax(axis=1), "ql": ql_uf.max(axis=1)})
print(topo.sort_values("ql", ascending=False).round(2).head(10))

## 8. Recortes econômicos

**Pergunta:** MATOPIBA, Semiárido e Metrópoles têm economias diferentes do restante?

### 8.1 Tabela comparativa

In [ ]:
def resumo(df):
    """Resumir indicadores de um grupo de municípios."""
    vab = df["vab_total"].sum()
    out = {
        "municípios": len(df),
        "população (mi)": df["populacao"].sum() / 1e6,
        "% do PIB": df["pib"].sum() / mun["pib"].sum() * 100,
        "PIB per capita (R$)": df["pib"].sum() * 1000 / df["populacao"].sum(),
        "mediana per capita (R$)": df["pib_per_capita"].median(),
    }

    # Calcular composição setorial ponderada (%)
    for s, n in zip(SETORES, NOMES_SETORES):
        out[n + " (%)"] = df[s].sum() / vab * 100

    out["HHI mediano"] = df["hhi"].median()
    return pd.Series(out)


# Montar tabela comparativa (GRUPOS: None = Brasil inteiro)
tabela = pd.DataFrame(
    {nome: resumo(mun[mun[col]] if col else mun) for nome, col in GRUPOS.items()}
).T
print(tabela.round(2).to_string())

### 8.2 HHI: mediana do município × HHI do agregado

In [ ]:
# Mostrar o HHI mediano com 3 casas decimais
with pd.option_context("display.float_format", "{:.3f}".format):
    print(tabela[["HHI mediano"]])

# Somar os setores de cada grupo
soma = pd.DataFrame(
    {
        nome: mun.loc[mun[col], SETORES].sum() if col else mun[SETORES].sum()
        for nome, col in GRUPOS.items()
    }
).T

# Calcular o HHI do agregado de cada grupo
print(hhi(soma).round(3))

### 8.3 Gráfico de composição

In [ ]:
# Selecionar a composição setorial de cada grupo
comp = tabela[[n + " (%)" for n in NOMES_SETORES]].copy()
comp.columns = NOMES_SETORES

# Criar barras empilhadas
fig = px.bar(
    comp,
    x=comp.index,
    y=NOMES_SETORES,
    color_discrete_map=CORES_SETORES,
    title="Composição do VAB por recorte (2021)",
    template="plotly_white",
    labels={"value": "% do VAB", "variable": "Setor"},
)
fig.update_xaxes(title="Recorte")
fig.show()

## 9. Perfis econômicos (agrupamento)

**Pergunta:** os municípios formam grupos naturais pela composição setorial?

### 9.1 Escolher o número de grupos (k)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Excluir municípios com VAB negativo
base = mun.loc[~mun["vab_negativo"]].copy()

# Calcular participações (fração do total) e contar quase-zeros
p = participacoes(base)
print((p < 0.001).sum())

# Trocar valores muito pequenos por um mínimo e renormalizar
p = p.clip(lower=0.001)
p = p.div(p.sum(axis=1), axis=0)

# Aplicar a transformação CLR
logp = np.log(p)
clr = logp.sub(logp.mean(axis=1), axis=0)

# Padronizar as variáveis
X = StandardScaler().fit_transform(clr)

# Testar k de 2 a 8
resultados = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=3000, random_state=42)
    resultados.append((k, km.inertia_, sil))

print(pd.DataFrame(resultados, columns=["k", "inércia", "silhueta"]).round(3))

### 9.2 Caracterizar os perfis

In [ ]:
# Contar municípios por grupo (o de VAB negativo fica vazio)
print(mun["grupo"].value_counts(dropna=False).sort_index())

# Mostrar o nome atribuído a cada grupo
print(mun.groupby("grupo")["perfil"].first())

# Definir colunas usadas no perfil
cols = ["vab_total", "populacao", "pib", "pib_per_capita"] + SETORES


def perfil(df):
    """Resumir um grupo de municípios."""
    vab = df["vab_total"].sum()
    out = {
        "municípios": len(df),
        "% população": df["populacao"].sum() / mun["populacao"].sum() * 100,
        "% PIB": df["pib"].sum() / mun["pib"].sum() * 100,
        "PIB per capita (R$)": df["pib"].sum() * 1000 / df["populacao"].sum(),
        "mediana per capita (R$)": df["pib_per_capita"].median(),
    }

    # Calcular composição setorial ponderada (%)
    for s, n in zip(SETORES, NOMES_SETORES):
        out[n + " (%)"] = df[s].sum() / vab * 100
    return pd.Series(out)


# Montar tabela de perfis
perfis = mun.groupby("grupo")[cols].apply(perfil)
print(perfis.round(1).to_string())

### 9.3 Onde estão os perfis?

In [ ]:
# Calcular a distribuição regional de cada grupo (% dos municípios do grupo)
reg_grupo = pd.crosstab(mun["grupo"], mun["regiao"], normalize="index") * 100
print(reg_grupo[ORDEM_REGIOES].round(1))

# Calcular a distribuição dos grupos dentro de cada recorte
for nome, col in [("MATOPIBA", "matopiba"), ("Semiárido", "semiarido"), ("Metrópoles", "em_metropole")]:
    dist = mun.loc[mun[col], "grupo"].value_counts(normalize=True).sort_index() * 100
    print(nome, dist.round(1).to_dict())

### 9.4 Testar a estabilidade

In [ ]:
from sklearn.metrics import adjusted_rand_score

from src.agrupamento import agrupar

# Reagrupar com outra semente
outro = agrupar(mun, k=K_PERFIS, semente=7)

# Comparar com o agrupamento da base
print(round(adjusted_rand_score(mun.loc[outro.index, "grupo"], outro), 3))

## 10. Quadrantes: concentração × renda

**Pergunta:** municípios concentrados em poucos setores são mais pobres?

### 10.1 Limites e correlação

In [ ]:
# Definir limites pelas medianas municipais
lim_hhi = mun["hhi"].median()
lim_renda = mun["pib_per_capita"].median()
print(round(lim_hhi, 3), round(lim_renda))

# Calcular a correlação de Spearman entre HHI e PIB per capita
print(mun[["hhi", "pib_per_capita"]].corr(method="spearman").round(3))

### 10.2 Conferir a classificação

In [ ]:
# Classificar cada município em um quadrante
conc = mun["hhi"] > lim_hhi
rico = mun["pib_per_capita"] > lim_renda
calc = pd.Series(
    np.select(
        [conc & ~rico, conc & rico, ~conc & ~rico, ~conc & rico],
        [
            "Concentrado, baixa renda",
            "Concentrado, alta renda",
            "Diversificado, baixa renda",
            "Diversificado, alta renda",
        ],
        default="",
    ),
    index=mun.index,
).where(mun["hhi"].notna())

# Comparar com a coluna da base
igual = calc.fillna("").eq(mun["quadrante"].astype(object).fillna(""))
print("reprodução igual à base:", igual.all())

### 10.3 Resumir e cruzar os quadrantes

In [ ]:
# Resumir cada quadrante
resumo_q = mun.groupby("quadrante").agg(
    municipios=("cod", "count"),
    pop=("populacao", "sum"),
    pib=("pib", "sum"),
)
resumo_q["% população"] = resumo_q["pop"] / mun["populacao"].sum() * 100
resumo_q["% PIB"] = resumo_q["pib"] / mun["pib"].sum() * 100
print(resumo_q[["municipios", "% população", "% PIB"]].round(1))

# Cruzar quadrantes com perfil e região (% dos municípios de cada quadrante)
print((pd.crosstab(mun["quadrante"], mun["perfil"], normalize="index") * 100).round(1))
print((pd.crosstab(mun["quadrante"], mun["regiao"], normalize="index") * 100)[ORDEM_REGIOES].round(1))

# Conferir contagem (o município sem HHI fica sem quadrante)
print(mun["quadrante"].value_counts(dropna=False))

### 10.4 Dispersão

In [ ]:
# Criar dispersão: HHI x PIB per capita (escala log no eixo Y)
fig = px.scatter(
    mun.dropna(subset=["quadrante"]),
    x="hhi",
    y="pib_per_capita",
    color="quadrante",
    color_discrete_map=CORES_QUADRANTES,
    log_y=True,
    opacity=0.5,
    hover_name="nome",
    hover_data={"perfil": True, "populacao": ":,.0f"},
    title="Concentração setorial × PIB per capita dos municípios (2021)",
    template="plotly_white",
    height=650,
)

# Marcar os limites dos quadrantes
fig.add_vline(x=lim_hhi, line_dash="dash", line_color="gray")
fig.add_hline(y=lim_renda, line_dash="dash", line_color="gray")
fig.update_xaxes(title="HHI (0 = diversificado, 1 = um só setor)")
fig.update_yaxes(title="PIB per capita (R$, escala log)")
fig.show()

# Listar os 10 maiores municípios do quadrante de maior exposição
exp = mun[mun["quadrante"] == "Concentrado, baixa renda"]
cols = ["nome", "populacao", "pib_per_capita", "hhi", "perfil"]
print(exp.nlargest(10, "populacao")[cols].round(3))

## 11. Limitações

**Limitações registradas**
- Ano-base 2021 (último com composição setorial).
- Mineração e petróleo estão dentro de **indústria** (padrão IBGE).
- Município com VAB negativo (Cachoeira Dourada) fica sem participações, HHI, perfil e quadrante.
- Quadrantes por mediana são arbitrários.
- Recortes se sobrepõem.
- QL calculado com o Brasil como referência e não usado no nível municipal.